In [1]:
!pip -q install kagglehub timm scikit-learn pandas numpy pillow
!apt -y install git git-lfs >/dev/null
!git lfs install >/dev/null

In [2]:
import os
import glob
import random
from collections import OrderedDict

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
import kagglehub
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import top_k_accuracy_score, accuracy_score, classification_report

# Set Up

In [3]:
SEED = 42 # random seed for reproducibility

# url of the github repo and where to store it
REPO_URL = "https://github.com/Brandenn28/ML.git"
CLONE_DIR = "/content/ML"
if not os.path.exists(CLONE_DIR):
    !git clone --depth=1 "{REPO_URL}" "{CLONE_DIR}"
%cd "{CLONE_DIR}"
!git lfs pull # Pull large files that are tracked by git lfs

# Base dataset paths
DATA_ROOT = "/content/ML/AML_project_herbarium_dataset"
PROCESSED_DIR = os.path.join(DATA_ROOT, "processed") # processed csv files

# Image parameters
IMG_SIZE = 224 # input size for DINOv2
# normalisation mean and std
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Data loader parameters
BATCH_SIZE = 64 # batch size for feature extraction
NUM_WORKERS = 2 # number of worker processes

MODEL_ID = "juliostat/dinov2_patch14_reg4_onlyclassifier_then_all/PyTorch/default" # Kaggle DINOv2 model id
CACHE_DIR = PROCESSED_DIR # where to cache extracted features

Cloning into '/content/ML'...
remote: Enumerating objects: 5133, done.
remote: Counting objects: 100% (5133/5133), done.
remote: Compressing objects: 100% (5126/5126), done.
remote: Total 5133 (delta 7), reused 5128 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (5133/5133), 576.60 MiB | 24.29 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Updating files: 100% (4974/4974), done.
/content/ML


# Utility functions

In [4]:
# Set seeds for reproducibility
def set_seed(seed: int = 42) -> None:
  random.seed(seed) # set python random seed
  np.random.seed(seed) # set numpy random seed
  torch.manual_seed(seed) # set torch cpu seed
  if torch.cuda.is_available():
      torch.cuda.manual_seed_all(seed) # set torch gpu seed

# Check that the processed csv files exist
def check_processed_files() -> None:
  required = [ # list of csv files that must exist
      "species_mapping.csv",
      "train_v2.csv",
      "val_v2.csv",
      "test.csv",
      "test_with_groundtruth.csv",
  ]
  # build full paths and see which ones are missing
  missing = [f for f in required if not os.path.exists(os.path.join(PROCESSED_DIR, f))]
  if missing:
      raise FileNotFoundError(f"Missing processed files: {missing} in {PROCESSED_DIR}")
  print("DATA_ROOT:", DATA_ROOT)
  print("Processed dir:", PROCESSED_DIR)

# Build training and evaluation transforms
def build_transforms():
  # transform for train images
  train_tf = transforms.Compose([
      transforms.Resize(256), # resize shorter side to 256
      transforms.CenterCrop(IMG_SIZE), # center crop to 224 x 224
      transforms.ToTensor(), # convert to tensor
      transforms.Normalize(MEAN, STD), # normalize with ImageNet stats
      ])

  eval_tf = transforms.Compose([
      transforms.Resize(256),
      transforms.CenterCrop(IMG_SIZE),
      transforms.ToTensor(),
      transforms.Normalize(MEAN, STD),
      ])
  return train_tf, eval_tf

#Dataset

In [5]:
# Dataset that reads image paths and labels from a csv file
class CSVImageDataset(Dataset):
  def __init__(self, csv_path: str, transform=None):
      self.df = pd.read_csv(csv_path) # read the csv into a data frame

      # read relative paths if the column exists, else empty list
      rel_list = self.df["rel_path"].astype(str).tolist() if "rel_path" in self.df.columns else []
      # convert to full paths under DATA_ROOT
      rel_joined = [os.path.join(DATA_ROOT, p).replace("\\", "/") for p in rel_list]

      # read absolute paths if present
      if "abs_path" in self.df.columns:
          abs_list = self.df["abs_path"].astype(str).tolist()
      else:
          abs_list = ["" for _ in rel_joined] # otherwise dummy empty strings


      paths = []  # final list of paths to use
      for a, r in zip(abs_list, rel_joined):
          a_norm = a.replace("\\", "/") # normalize slashes
          # if abs path exists, use it, else use relative path
          p = a_norm if a_norm and os.path.exists(a_norm) else r
          paths.append(p)
      self.paths = paths

      # read labels if label_idx column exists
      self.labels = self.df["label_idx"].astype(int).tolist() if "label_idx" in self.df.columns else None
      self.transform = transform # store transform

  def __len__(self):
    return len(self.paths) # number of samples is number of paths

  def __getitem__(self, i):
    path = self.paths[i] # pick the i-th path
    img = Image.open(path).convert("RGB") # open image as rgb
    x = self.transform(img) if self.transform else img # apply transform if given
    if self.labels is None: # if no labels, just return image and path
        return x, path
    return x, self.labels[i], path # otherwise return image, label, path

# DINOv2 backbone loaded from Kaggle Models

In [6]:
# Small helper to guess number of extra tokens and grid size from pos_embed length
def _infer_extra_tokens_and_grid_len(total_tokens: int):
  for e in (0, 1, 2, 5): # try a few possible extra token counts
      n = total_tokens - e # remaining tokens for grid
      r = int(round(n ** 0.5)) # grid side length guess
      if r * r == n: # if perfect square, accept this
          return e, r
  r = int(round(total_tokens ** 0.5)) # fallback, assume no extra tokens
  return 0, r

# Load the plant specific DINOv2 model and build a feature extractor
def load_plant_dinov2(device: torch.device):
  print("Downloading Kaggle DINOv2 model...")
  model_dir = kagglehub.model_download(MODEL_ID) # download model files
  print("Model directory:", model_dir)

  # Search for a checkpoint file with known extensions
  ckpt_candidates = []
  for pat in ("*.pt", "*.pth", "*.bin", "*.safetensors", "*.pth.tar"):
      ckpt_candidates.extend(glob.glob(os.path.join(model_dir, "**", pat), recursive=True))
  if not ckpt_candidates: # if none found, raise a error
      raise FileNotFoundError("No checkpoint file found in the Kaggle model folder.")
  ckpt_candidates.sort() # sort for deterministic choice
  ckpt_path = ckpt_candidates[0] # pick the first one
  print("Checkpoint chosen:", ckpt_path)

  # Import argparse to register its Namespace for safe loading
  import argparse
  try:
      torch.serialization.add_safe_globals([argparse.Namespace]) # allow argparse.Namespace in checkpoint
  except Exception:
      pass # ignore if this is not needed

  # Try to load checkpoint in a safe way
  try:
      sd = torch.load(ckpt_path, map_location="cpu")
  except Exception as e:
      print("Safe load failed, retrying with weights_only=False:", e)
      sd = torch.load(ckpt_path, map_location="cpu", weights_only=False)

  # Unwrap if weights are nested under common keys
  for k in ["state_dict", "model", "net", "params"]:
      if isinstance(sd, dict) and k in sd:
          sd = sd[k]
  if not isinstance(sd, dict):        # if still not a dict, checkpoint format is wrong
      raise TypeError(f"Unexpected checkpoint format: {type(sd)}")

  # Strip common prefixes
  clean_sd = OrderedDict()
  for k, v in sd.items():
      nk = k.replace("module.", "").replace("backbone.", "").replace("model.", "")
      clean_sd[nk] = v

  # Decide which timm DINOv2 variant to use from cls_token dimension
  if "cls_token" not in clean_sd:
      raise KeyError("cls_token not found in checkpoint keys.")
  embed_dim = clean_sd["cls_token"].shape[-1] # dimension of embedding
  if embed_dim == 768:
      model_name = "vit_base_patch14_dinov2" # base model
  elif embed_dim == 1024:
      model_name = "vit_large_patch14_dinov2.lvd142m" # large model
  else:
      raise ValueError(f"Unexpected embed dim {embed_dim}")

  # Create the timm model without classifier head
  backbone = timm.create_model(
      model_name,
      pretrained=False, # we will load our own weights
      num_classes=0, # no classification head, only features
      img_size=IMG_SIZE, # image size
      dynamic_img_size=False, # disable dynamic size
  )
  print("Backbone created:", model_name)

  # Copy state dict so we can edit it
  load_sd = clean_sd.copy()
  # Handle positional embedding resizing when shape does not match
  if "pos_embed" in load_sd and hasattr(backbone, "pos_embed"):
      pe_sd = load_sd["pos_embed"] # positional embedding from checkpoint, shape [1, T_sd, C]
      pe_m = backbone.pos_embed # positional embedding from model
      if isinstance(pe_m, torch.nn.Parameter):
          pe_m = pe_m.data # get underlying tensor if parameter

      T_sd = pe_sd.shape[1] # number of tokens in checkpoint
      T_m = pe_m.shape[1] # number of tokens in model

      # Infer how many extra tokens and grid size in checkpoint and model
      e_sd, gs_sd = _infer_extra_tokens_and_grid_len(T_sd)
      e_m, gs_m = _infer_extra_tokens_and_grid_len(T_m)

      # Split checkpoint embed into extra and grid tokens
      extra_sd = pe_sd[:, :e_sd] if e_sd > 0 else pe_sd[:, :0]
      grid_sd = pe_sd[:, e_sd:]
      # Split model embed into extra and grid tokens
      extra_m = pe_m[:, :e_m] if e_m > 0 else pe_m[:, :0]
      grid_m = pe_m[:, e_m:]

      C = pe_sd.shape[-1] # embedding dimension

      # Reshape checkpoint grid to [1, C, H, W] for interpolation
      grid_sd = grid_sd.reshape(1, gs_sd, gs_sd, C).permute(0, 3, 1, 2)
      # Resize checkpoint grid to match model grid size
      grid_sd = torch.nn.functional.interpolate(
          grid_sd,
          size=(gs_m, gs_m),
          mode="bicubic",
          align_corners=False,
      )
      # Reshape back to [1, H*W, C]
      grid_sd = grid_sd.permute(0, 2, 3, 1).reshape(1, gs_m * gs_m, C)

      # Combine model extra tokens with resized checkpoint grid
      load_sd["pos_embed"] = torch.cat([extra_m, grid_sd], dim=1)

  # Remove register tokens that the timm backbone may not have
  for k in list(load_sd.keys()):
      if k.startswith("reg_token"):
          load_sd.pop(k)

  # Load weights into the timm backbone
  missing, unexpected = backbone.load_state_dict(load_sd, strict=False)
  print(f"Loaded DINOv2 weights with missing={len(missing)}, unexpected={len(unexpected)}")

  # Freeze all parameters so we only use the model as a feature extractor
  for p in backbone.parameters():
      p.requires_grad = False

  # Move model to device and set eval mode
  backbone = backbone.to(device).eval()
  return backbone

# Feature extraction with caching

In [7]:
# Create a DataLoader for a given csv and transform
def build_loader(csv_path: str, transform, batch_size: int, shuffle: bool = False):
  ds = CSVImageDataset(csv_path, transform=transform) # build dataset
  return DataLoader(
      ds,
      batch_size=batch_size,
      shuffle=shuffle,
      num_workers=NUM_WORKERS,
      pin_memory=torch.cuda.is_available(),
  )

# Extract features and labels from a DataLoader
@torch.no_grad()
def extract_features(backbone, loader, device):
  feats, labels, rel_paths = [], [], [] # lists to store results
  for xb, yb, paths in loader: # loop over batches
      xb = xb.to(device) # move images to device
      f = backbone(xb).detach().cpu() # run backbone and move to cpu
      feats.append(f) # store features
      labels.append(torch.tensor(yb)) # store labels
      rel_paths.extend(paths) # store paths
  feats = torch.cat(feats, 0).numpy() # stack all features
  labels = torch.cat(labels, 0).numpy() # stack all labels
  return feats, labels, np.array(rel_paths) # return arrays

# Load cached features if available, otherwise compute and cache them
def get_or_build_features(split_name: str, csv_path: str, backbone, transform, device):
  X_path = os.path.join(CACHE_DIR, f"{split_name}_X.npy") # feature cache path
  y_path = os.path.join(CACHE_DIR, f"{split_name}_y.npy") # label cache path
  paths_path = os.path.join(CACHE_DIR, f"{split_name}_paths.npy") # rel path cache path

  # If cache files exist, load and return them
  if os.path.exists(X_path) and os.path.exists(y_path):
      print(f"[{split_name}] Loading cached features...")
      X = np.load(X_path)
      y = np.load(y_path)
      rel_paths = np.load(paths_path, allow_pickle=True) if os.path.exists(paths_path) else None
      return X, y, rel_paths

  # Otherwise build loader and extract features
  print(f"[{split_name}] Cache not found. Extracting features...")
  loader = build_loader(csv_path, transform, BATCH_SIZE, shuffle=False)
  X, y, rel_paths = extract_features(backbone, loader, device)

  # Save arrays to disk for future runs
  np.save(X_path, X)
  np.save(y_path, y)
  if rel_paths is not None:
      np.save(paths_path, rel_paths)

  return X, y, rel_paths


# Extract features for test set, which has no labels in its csv
@torch.no_grad()
def extract_test_features(backbone, test_df: pd.DataFrame, transform, device):
  # Local dataset class for test data
  class TestDataset(Dataset):
      def __init__(self, df, root_dir, transform):
          self.df = df.reset_index(drop=True) # store a clean copy of df
          self.root_dir = root_dir # root directory of images
          self.transform = transform # image transform

      def __len__(self):
          return len(self.df) # number of rows in df

      def __getitem__(self, i):
          row = self.df.iloc[i] # get i-th row
          rel_p = row["rel_path"] # relative path
          full_path = os.path.join(self.root_dir, rel_p).replace("\\", "/")
          # if main path does not exist and abs_path exists, use that
          if not os.path.exists(full_path) and "abs_path" in row and isinstance(row["abs_path"], str):
              alt = row["abs_path"]
              if os.path.exists(alt):
                  full_path = alt
          img = Image.open(full_path).convert("RGB") # open image
          x = self.transform(img) # apply transform
          return x, rel_p # return tensor and rel path

  ds = TestDataset(test_df, DATA_ROOT, transform) # create dataset
  loader = DataLoader(
      ds,
      batch_size=BATCH_SIZE,
      shuffle=False,
      num_workers=NUM_WORKERS,
      pin_memory=torch.cuda.is_available(),
  )

  feats, rel_paths = [], [] # store features and paths
  for xb, rel in loader: # loop over test loader
      xb = xb.to(device) # move images to device
      f = backbone(xb).detach().cpu() # extract features
      feats.append(f) # collect features
      rel_paths.extend(rel) # collect rel paths
  feats = torch.cat(feats, 0).numpy() # stack features
  rel_paths = np.array(rel_paths) # convert paths to array
  return feats, rel_paths # return arrays

#Classifier training and evaluation

In [8]:
# Train logistic regression on top of DINOv2 features
def train_logreg_classifier(train_X, train_y):
  weights_csv = os.path.join(PROCESSED_DIR, "train_sample_weights.csv")  # path to sample weights
  sample_weight = None # default is no weights

  if os.path.exists(weights_csv): # if weights file exists
      wdf = pd.read_csv(weights_csv) # read weights
      # check that length matches and column exists
      if len(wdf) == len(train_X) and "sample_weight" in wdf.columns:
          sample_weight = wdf["sample_weight"].values
          print("Using sample weights from train_sample_weights.csv")
      else:
          print("Found train_sample_weights.csv but shape or column mismatch, ignoring weights")

  # Build multinomial logistic regression classifier
  clf = LogisticRegression(
      penalty="l2", # L2 regularization
      solver="saga", # saga solver supports multinomial and sample weights
      multi_class="multinomial", # use true multinomial softmax
      max_iter=2000, # number of optimisation iterations
      n_jobs=-1, # use all cpu cores
      verbose=0, # no solver output
  )
  # Fit classifier on training features and labels
  clf.fit(train_X, train_y, sample_weight=sample_weight)
  return clf

# Evaluate classifier on a given split
def evaluate_on_split(name: str, clf, X, y):
  prob = clf.predict_proba(X) # get predicted probabilities for each class
  pred = prob.argmax(1) # convert to predicted class index

  top1 = accuracy_score(y, pred) # compute top 1 accuracy
  top5 = top_k_accuracy_score(y, prob, k=5, labels=np.arange(prob.shape[1])) # compute top 5 accuracy

  print(f"[{name}] Top-1: {top1:.4f}, Top-5: {top5:.4f}")
  return top1, top5 # return accuracies

# Evaluate classifier on test set with ground truth file
def evaluate_on_test(clf, test_X, test_rel_paths, gt_df: pd.DataFrame):
  test_prob = clf.predict_proba(test_X) # predicted probability for each test image

  # Build dict from rel_path to probability vector
  pred_lookup = {rp: prob for rp, prob in zip(test_rel_paths, test_prob)}

  y_true, y_pred_top1, y_pred_top5_ok = [], [], [] # lists for metrics
  missing = 0 # counter for missing paths

  # Iterate over ground truth rows
  for _, row in gt_df.iterrows():
      rp = row["rel_path"] # rel path in ground truth
      true_label = int(row["label_idx"]) # true label index

      if rp not in pred_lookup: # if this image has no prediction
          missing += 1 # count it
          continue

      probs = pred_lookup[rp] # probability vector for this path
      pred_label = probs.argmax() # top 1 prediction
      top5_labels = np.argsort(-probs)[:5] # top 5 predicted labels

      y_true.append(true_label) # collect true label
      y_pred_top1.append(pred_label) # collect predicted label
      y_pred_top5_ok.append(int(true_label in top5_labels)) # 1 if correct in top 5

  if missing > 0:
      print(f"[test] Warning: {missing} ground truth images were missing predictions.")

  y_true = np.array(y_true) # convert to array
  y_pred_top1 = np.array(y_pred_top1) # convert to array
  y_pred_top5_ok = np.array(y_pred_top5_ok) # convert to array

  top1 = (y_true == y_pred_top1).mean() # compute top 1 accuracy
  top5 = y_pred_top5_ok.mean() # compute top 5 accuracy

  print(f"[test] Top-1 Accuracy: {top1:.4f}")
  print(f"[test] Top-5 Accuracy: {top5:.4f}")
  print("\n[test] Classification report (Top-1 predictions):")
  print(classification_report(y_true, y_pred_top1, digits=3))

  # Save simple prediction csv for later use
  out_path = os.path.join(PROCESSED_DIR, "test_predictions.csv")
  pd.DataFrame({
      "rel_path": test_rel_paths, # each test image path
      "pred_label_idx": test_prob.argmax(1), # predicted top 1 label for each
  }).to_csv(out_path, index=False)
  print(f"[test] Saved predictions to {out_path}")


#Main

In [9]:
def main():
  set_seed(SEED) # set random seeds
  check_processed_files() # make sure csvs exist
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # pick device
  print("Device:", device)

  train_tf, eval_tf = build_transforms() # get transforms

  # Load DINOv2 backbone
  backbone = load_plant_dinov2(device)

  # Build full paths to split csv files
  train_csv = os.path.join(PROCESSED_DIR, "train_v2.csv")
  val_csv = os.path.join(PROCESSED_DIR, "val_v2.csv")
  test_csv = os.path.join(PROCESSED_DIR, "test.csv")
  test_gt_csv = os.path.join(PROCESSED_DIR, "test_with_groundtruth.csv")

  # Extract or load cached features for train and val
  train_X, train_y, _ = get_or_build_features("train", train_csv, backbone, train_tf, device)
  val_X, val_y, _ = get_or_build_features("val", val_csv, backbone, eval_tf, device)

  # Train logistic regression classifier on train features
  clf = train_logreg_classifier(train_X, train_y)

  # Evaluate on validation set
  evaluate_on_split("val", clf, val_X, val_y)

  val_pred = clf.predict(val_X)  # predicted top-1 label for each val sample
  print("\n[val] Classification report (Top-1 predictions):")
  print(classification_report(val_y, val_pred, digits=3))

  # Paths to cached test features
  test_X_path = os.path.join(CACHE_DIR, "test_X.npy")
  test_rel_paths_path = os.path.join(CACHE_DIR, "test_rel_paths.npy")

  # Read raw test csv
  test_df = pd.read_csv(test_csv)
  # If cache exists, load it, otherwise extract features and cache them
  if os.path.exists(test_X_path) and os.path.exists(test_rel_paths_path):
      print("[test] Loading cached features...")
      test_X = np.load(test_X_path)
      test_rel_paths = np.load(test_rel_paths_path, allow_pickle=True)
  else:
      print("[test] Cache not found, extracting features...")
      test_X, test_rel_paths = extract_test_features(backbone, test_df, eval_tf, device)
      np.save(test_X_path, test_X)
      np.save(test_rel_paths_path, test_rel_paths)

  # Read ground truth csv for test set
  gt_df = pd.read_csv(test_gt_csv)
  # Evaluate classifier on test set
  evaluate_on_test(clf, test_X, test_rel_paths, gt_df)


if __name__ == "__main__":
    main()



DATA_ROOT: /content/ML/AML_project_herbarium_dataset
Processed dir: /content/ML/AML_project_herbarium_dataset/processed
Device: cuda
Model directory: /kaggle/input/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3
Checkpoint chosen: /kaggle/input/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3/model_best.pth.tar
Backbone created: vit_base_patch14_dinov2
Loaded DINOv2 weights with missing=0, unexpected=2
[train] Cache not found. Extracting features...


/tmp/ipython-input-3254810421.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels.append(torch.tensor(yb)) # store labels


[val] Cache not found. Extracting features...


/tmp/ipython-input-3254810421.py:20: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels.append(torch.tensor(yb)) # store labels


Using sample weights from train_sample_weights.csv


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[val] Top-1: 0.7162, Top-5: 0.8903

[val] Classification report (Top-1 predictions):
              precision    recall  f1-score   support

           0      0.750     1.000     0.857         6
           1      0.818     0.900     0.857        10
           2      0.500     0.600     0.545        10
           3      0.143     0.100     0.118        10
           4      0.714     1.000     0.833        10
           5      0.636     0.700     0.667        10
           6      0.500     0.600     0.545        10
           7      0.417     0.500     0.455        10
           8      0.700     0.700     0.700        10
           9      0.600     0.600     0.600        10
          10      0.545     0.600     0.571        10
          11      0.500     0.200     0.286        10
          12      0.500     0.500     0.500        10
          13      0.667     0.600     0.632        10
          14      0.800     0.800     0.800        10
          15      0.692     0.900     0.783       

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
